# How To: Process Time-Series for Growth Curves

Process a series of plate images taken at different time points,
extract measurements from each, and combine them into a time-series
DataFrame suitable for growth curve analysis.

In [ ]:
import phenotypic as pht
from phenotypic.data import load_plate_12hr, load_plate_72hr
from phenotypic.enhance import GaussianBlur, EnhanceLocalContrast
from phenotypic.detect import OtsuDetector
from phenotypic.measure import MeasureSize
import pandas as pd

## Load a Time Series

PhenoTypic includes a sample plate series captured across 6 time points.

In [ ]:
series = [load_plate_12hr(mode="GridImage"), load_plate_72hr(mode="GridImage")]
print(f"Loaded {len(series)} time points")

## Process Each Time Point

In [ ]:
pipeline = pht.ImagePipeline(
    ops=[GaussianBlur(sigma=2.0), EnhanceLocalContrast(clip_limit=0.01), OtsuDetector()],
    meas=[MeasureSize()],
)

all_measurements = []
for i, plate in enumerate(series):
    df = pipeline.apply_and_measure(plate)
    df["TimePoint"] = i
    all_measurements.append(df)

combined = pd.concat(all_measurements, ignore_index=True)
print(f"Total measurements: {len(combined)} colonies across {len(series)} time points")
combined.head()

## Summarize per Time Point

Group by time point to see how colony size evolves.

In [ ]:
summary = combined.groupby("TimePoint")["Size_Area"].agg(["mean", "std", "count"])
summary

For logistic growth model fitting, see the
`LogGrowthModel` class in `phenotypic.analysis`.